# Micro-CT Digital Core Pipeline

Единый ноутбук для всего пайплайна — от подготовки данных до валидации сегментации.

### Как пользоваться

- **Каждая секция самодостаточна**: свои импорты, свой конфиг, свои чекпоинты.
- **Секции можно пропускать**: если данные уже подготовлены — иди к Section 2;
  если сети уже извлечены — иди к Section 4.
- **Для Colab**: скопируй ноутбук, установи зависимости (`pip install -r src/requirements.txt`)
  и запускай нужные секции.

### Содержание

0. **Setup & Environment Check** — импорты, пути, проверка зависимостей
1. **Prepare Data** — сканирование rock-папок, запись индексных CSV (однократно)
2. **Train Segmentation Model** — обучение TopologyAdaptiveRoutedUNet3D
3. **Extract Pore Networks** — PoreSpy/OpenPNM → .pt файлы
4. **Train GNN Permeability Model** — обучение GNN на .pt сетях
5. **Run Full Pipeline** — сегментация + экстракция + GNN на одном кубе
6. **Validate Segmentation** — Dice, BCE, error rate по группам
7. **(Optional) Compare Model Variants** — topology vs adaptive на 64³

---
## 0. Setup & Environment Check

Эту ячейку нужно выполнить **всегда первой** — она настраивает пути и проверяет зависимости.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
for candidate in (ROOT, *ROOT.parents):
    if (candidate / "src" / "utils").is_dir():
        ROOT = candidate
        break
else:
    raise RuntimeError("Project root with src/utils was not found")

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print("ROOT:", ROOT)

In [ ]:
import torch

from utils import check_required_dependencies

status = check_required_dependencies()
print(status)
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

---
## 1. Prepare Data

Сканирует rock-папки в `data/` и записывает индексные CSV-файлы в `datasets/`.
**Запускать один раз** при добавлении новой породы или изменении состава данных.

Если `datasets/` уже заполнены — **пропусти эту секцию**.

In [ ]:
import pandas as pd

from utils import (
    DEFAULT_CUBE_SIZES,
    discover_rock_volumes,
    write_patch_indices,
)

In [ ]:
DATA_ROOT = ROOT / "data"
INDEX_ROOT = ROOT / "datasets"
CUBE_SIZES = list(DEFAULT_CUBE_SIZES)
SHAPE = (1000, 1000, 1000)
ROCKS = None  # или ["Berea", "BanderaBrown"]
VAL_FRACTION = 0.2
SEED = 42
COMPUTE_AUX_TARGETS = True
PORE_VALUE = 0
STRIDE_BY_SIZE = {64: 64, 128: 128, 192: 192}

print("cube sizes:", CUBE_SIZES)

In [ ]:
specs = discover_rock_volumes(ROOT, data_root=DATA_ROOT, index_root=INDEX_ROOT, rocks=ROCKS, shape=SHAPE)
pd.DataFrame([
    {
        "rock": spec.name,
        "data_dir": str(spec.data_dir),
        "index_dir": str(INDEX_ROOT / spec.name),
        "gray": str(spec.gray_path),
        "binary": str(spec.binary_path),
        "shape": spec.shape,
    }
    for spec in specs
])

In [ ]:
summary = write_patch_indices(
    ROOT,
    data_root=DATA_ROOT,
    index_root=INDEX_ROOT,
    rocks=ROCKS,
    cube_sizes=CUBE_SIZES,
    shape=SHAPE,
    stride_by_size=STRIDE_BY_SIZE,
    val_fraction=VAL_FRACTION,
    seed=SEED,
    compute_aux_targets=COMPUTE_AUX_TARGETS,
    pore_value=PORE_VALUE,
)

summary.groupby(["rock", "cube_size", "split"]).size().rename("samples").reset_index()

---
## 2. Train Segmentation Model

Обучает TopologyAdaptiveRoutedUNet3D на raw-кубах → бинарные маски пор.
Сохраняет лучший чекпоинт в `models/segmentation_best.pth`.
При `TRAIN_MODE = "quick"` — обучение с ограничением по батчам (для итераций).
При `TRAIN_MODE = "full"` — полная эпоха.

In [ ]:
import importlib
from torch.utils.data import DataLoader
from tqdm import tqdm

import utils.common as common_module
import utils.data as data_module
import utils.adaptive_routing as adaptive_module
import utils.losses as losses_module

common_module = importlib.reload(common_module)
data_module = importlib.reload(data_module)
adaptive_module = importlib.reload(adaptive_module)
losses_module = importlib.reload(losses_module)

BCEDiceLoss = losses_module.BCEDiceLoss
DEFAULT_CUBE_SIZES = data_module.DEFAULT_CUBE_SIZES
BereaPatchDataset = data_module.BereaPatchDataset
CubeSizeBatchSampler = data_module.CubeSizeBatchSampler
TopologyAdaptiveRoutedUNet3D = adaptive_module.TopologyAdaptiveRoutedUNet3D
auxiliary_physics_loss = losses_module.auxiliary_physics_loss
dice_score_from_logits = losses_module.dice_score_from_logits
topology_prediction_loss = losses_module.topology_prediction_loss
from utils.training import EarlyStopping, MetricTracker
from utils.topology import TOPOLOGY_FEATURE_DIM

In [ ]:
TRAIN_MODE = "quick"  # quick | full
CUBE_SIZES = list(DEFAULT_CUBE_SIZES)
BATCH_SIZE_BY_CUBE_SIZE = {64: 2, 128: 1, 192: 1}
EPOCHS = 10
LR = 1e-4
AUX_WEIGHT = 0.05
TOPOLOGY_WEIGHT = 0.01
PATIENCE = 3
MIN_DELTA = 1e-4
BASE_CHANNELS = 16
CTX_DIM = 64
PH_DIM = TOPOLOGY_FEATURE_DIM
SAMPLES_PER_GROUP = 8 if TRAIN_MODE == "quick" else None
MAX_TRAIN_BATCHES = 64 if TRAIN_MODE == "quick" else None
MAX_VAL_BATCHES = 16 if TRAIN_MODE == "quick" else None
SIZE_SAMPLING_WEIGHTS = {64: 0.50, 128: 0.35, 192: 0.15}
NUM_WORKERS = 0
PIN_MEMORY = device == "cuda"
USE_AMP = device == "cuda"
SAVE_CHECKPOINT = True
CHECKPOINT_PATH = ROOT / "models" / "segmentation_best.pth"

In [ ]:
MODEL_DIR = ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

train_ds = BereaPatchDataset(
    ROOT,
    split="train",
    cube_size=CUBE_SIZES,
    use_raw_gray=False,
    balance=True,
    samples_per_group=SAMPLES_PER_GROUP,
    size_sampling_weights=SIZE_SAMPLING_WEIGHTS,
    return_topology=True,
)
val_ds = BereaPatchDataset(
    ROOT,
    split="val",
    cube_size=CUBE_SIZES,
    use_raw_gray=False,
    noise_types=["none"],
    balance=False,
    samples_per_group=SAMPLES_PER_GROUP,
    return_topology=True,
)

train_sampler = CubeSizeBatchSampler(train_ds, BATCH_SIZE_BY_CUBE_SIZE, shuffle=True, seed=42)
val_sampler = CubeSizeBatchSampler(val_ds, BATCH_SIZE_BY_CUBE_SIZE, shuffle=False, seed=42)
train_loader = DataLoader(train_ds, batch_sampler=train_sampler, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
val_loader = DataLoader(val_ds, batch_sampler=val_sampler, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
print("mode:", TRAIN_MODE, "cube sizes:", CUBE_SIZES)
print("train:", len(train_ds), "val:", len(val_ds), "batches:", len(train_loader), len(val_loader))

In [ ]:
model = TopologyAdaptiveRoutedUNet3D(
    in_channels=1,
    out_channels=1,
    base_channels=BASE_CHANNELS,
    ctx_dim=CTX_DIM,
    ph_dim=PH_DIM,
    topology_dim=PH_DIM,
).to(device)
criterion = BCEDiceLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
print("parameters:", sum(p.numel() for p in model.parameters()))

In [ ]:
early_stopping = EarlyStopping(patience=PATIENCE, min_delta=MIN_DELTA, mode="min")
history = []
scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_stats = MetricTracker()
    train_bar = tqdm(train_loader, desc=f"train {epoch}/{EPOCHS}", leave=False)
    for batch_idx, batch in enumerate(train_bar):
        if MAX_TRAIN_BATCHES is not None and batch_idx >= MAX_TRAIN_BATCHES:
            break
        x = batch["x"].to(device)
        y = batch["y"].to(device)
        ph_features = batch.get("ph_features")
        topology_target = batch.get("topology_target")
        if ph_features is not None:
            ph_features = ph_features.to(device)
        if topology_target is not None:
            topology_target = topology_target.to(device)

        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            out = model(x, ph_features=ph_features, return_dict=True)
            logits = out["logits"]
            seg_loss, bce_loss, dice_loss = criterion(logits, y)
            aux_loss, aux_parts = auxiliary_physics_loss(
                out, y,
                porosity_target=batch["porosity"].to(device),
                percolation_target=batch["percolates"].to(device),
                porosity_weight=AUX_WEIGHT,
                percolation_weight=AUX_WEIGHT,
            )
            topo_loss, topo_parts = topology_prediction_loss(
                out, topology_target, topology_weight=TOPOLOGY_WEIGHT
            )
            loss = seg_loss + aux_loss + topo_loss
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        batch_size = x.size(0)
        with torch.no_grad():
            dice = dice_score_from_logits(logits, y)
        train_stats.update("loss", float(loss.detach().cpu()), batch_size)
        train_stats.update("seg_loss", float(seg_loss.detach().cpu()), batch_size)
        train_stats.update("dice", float(dice.detach().cpu()), batch_size)
        train_bar.set_postfix(train_stats.postfix("loss", "dice"))

    model.eval()
    val_stats = MetricTracker()
    with torch.no_grad():
        val_bar = tqdm(val_loader, desc=f"val {epoch}/{EPOCHS}", leave=False)
        for batch_idx, batch in enumerate(val_bar):
            if MAX_VAL_BATCHES is not None and batch_idx >= MAX_VAL_BATCHES:
                break
            x = batch["x"].to(device)
            y = batch["y"].to(device)
            ph_features = batch.get("ph_features")
            topology_target = batch.get("topology_target")
            if ph_features is not None:
                ph_features = ph_features.to(device)
            if topology_target is not None:
                topology_target = topology_target.to(device)

            with torch.cuda.amp.autocast(enabled=USE_AMP):
                out = model(x, ph_features=ph_features, return_dict=True)
                logits = out["logits"]
                seg_loss, bce_loss, dice_loss = criterion(logits, y)
                aux_loss, aux_parts = auxiliary_physics_loss(
                    out, y,
                    porosity_target=batch["porosity"].to(device),
                    percolation_target=batch["percolates"].to(device),
                    porosity_weight=AUX_WEIGHT,
                    percolation_weight=AUX_WEIGHT,
                )
                topo_loss, topo_parts = topology_prediction_loss(
                    out, topology_target, topology_weight=TOPOLOGY_WEIGHT
                )
                loss = seg_loss + aux_loss + topo_loss
            dice = dice_score_from_logits(logits, y)

            batch_size = x.size(0)
            val_stats.update("loss", float(loss.detach().cpu()), batch_size)
            val_stats.update("dice", float(dice.detach().cpu()), batch_size)
            val_bar.set_postfix(val_stats.postfix("loss", "dice"))

    train_metrics = train_stats.as_dict()
    val_metrics = val_stats.as_dict()
    history.append({"epoch": epoch, "train_loss": train_metrics["loss"], "train_dice": train_metrics["dice"], "val_loss": val_metrics["loss"], "val_dice": val_metrics["dice"]})
    print(f"epoch={epoch} train_loss={train_metrics['loss']:.4f} train_dice={train_metrics['dice']:.4f} val_loss={val_metrics['loss']:.4f} val_dice={val_metrics['dice']:.4f}")

    improved = early_stopping.step(val_metrics["loss"], epoch=epoch)
    if improved and SAVE_CHECKPOINT:
        torch.save({
            "model_state_dict": model.state_dict(),
            "epoch": epoch,
            "val_loss": val_metrics["loss"],
            "val_dice": val_metrics["dice"],
            "history": history,
            "base_channels": BASE_CHANNELS,
            "ctx_dim": CTX_DIM,
            "ph_dim": PH_DIM,
            "topology_dim": PH_DIM,
        }, CHECKPOINT_PATH)
        print("saved:", CHECKPOINT_PATH)
    if early_stopping.should_stop:
        print(f"early stop at epoch={epoch}; best val_loss={early_stopping.best:.4f}")
        break

---
## 3. Extract Pore Networks

Использует PoreSpy/OpenPNM для извлечения поровых сетей из сегментированных кубов.
Сохраняет `.pt` файлы в `outputs/networks/`.

**Важно**: это самая медленная секция — PoreSpy SNOW + OpenPNM Stokes solver
занимают минуты на больших кубах. Для быстрой проверки используй `USE_TARGET_MASK=True`
(берёт ground truth маску, без сегментации).

In [ ]:
import pandas as pd
from torch.utils.data import DataLoader
from tqdm import tqdm

from utils import (
    DEFAULT_CUBE_SIZES,
    BereaPatchDataset,
    DigitalCorePipeline,
    TOPOLOGY_FEATURE_DIM,
    TopologyAdaptiveRoutedUNet3D,
)

In [ ]:
CUBE_SIZES = list(DEFAULT_CUBE_SIZES)
SPLIT = "train"
MAX_SAMPLES = 5  # уменьши для быстрой проверки
USE_TARGET_MASK = True  # True = ground truth, без загрузки сегмент. модели
THRESHOLD = 0.5
VOXEL_SIZE_M = 2.25e-6
SEG_CHECKPOINT = ROOT / "models" / "segmentation_best.pth"

In [ ]:
OUT_DIR = ROOT / "outputs" / "networks"
OUT_DIR.mkdir(parents=True, exist_ok=True)

dataset = BereaPatchDataset(ROOT, split=SPLIT, cube_size=CUBE_SIZES, use_raw_gray=False, noise_types=["none"], balance=True)
loader = DataLoader(dataset, batch_size=1, shuffle=False, num_workers=0)

seg_model = None
if not USE_TARGET_MASK:
    checkpoint = torch.load(SEG_CHECKPOINT, map_location=device)
    ph_dim = int(checkpoint.get("ph_dim", TOPOLOGY_FEATURE_DIM))
    seg_model = TopologyAdaptiveRoutedUNet3D(
        base_channels=checkpoint.get("base_channels", 16),
        ctx_dim=checkpoint.get("ctx_dim", 64),
        ph_dim=ph_dim,
        topology_dim=ph_dim,
    ).to(device)
    seg_model.load_state_dict(checkpoint["model_state_dict"])

pipeline = DigitalCorePipeline(
    segmentation_model=seg_model,
    device=device,
    threshold=THRESHOLD,
    voxel_size=VOXEL_SIZE_M,
    mu=1.0e-3,
)
print("cube sizes:", CUBE_SIZES)
print("samples:", len(dataset))

In [ ]:
rows = []

for sample_id, batch in enumerate(tqdm(loader, desc="extract networks")):
    if MAX_SAMPLES is not None and sample_id >= MAX_SAMPLES:
        break

    if USE_TARGET_MASK:
        cube = batch["y"][0, 0].numpy() > 0.5
        input_is_pore_mask = True
    else:
        cube = batch["x"][0].numpy()
        input_is_pore_mask = False

    cube_size = int(batch["cube_size"][0])
    domain_size = (cube_size * VOXEL_SIZE_M, cube_size * VOXEL_SIZE_M, cube_size * VOXEL_SIZE_M)
    rock = batch["rock"][0]

    try:
        result = pipeline.run_cube(
            cube,
            input_is_pore_mask=input_is_pore_mask,
            domain_size=domain_size,
            include_ph=True,
            compute_openpnm_baseline=True,
        )
    except Exception as exc:
        print(f"sample {sample_id} skipped: {exc}")
        continue

    coord = batch["coord"][0].tolist()
    result.network.metadata.update({
        "sample_id": sample_id,
        "coord": coord,
        "rock": rock,
        "cube_size": cube_size,
        "openpnm_k": result.permeability.k_openpnm,
    })
    out_path = OUT_DIR / f"network_{SPLIT}_{sample_id:04d}.pt"
    torch.save(result.network, out_path)

    row = {
        "sample_id": sample_id,
        "path": str(out_path),
        "rock": rock,
        "cube_size": cube_size,
        "z": coord[0],
        "y": coord[1],
        "x": coord[2],
        **result.network.metadata,
    }
    if result.permeability.k_openpnm is not None:
        row.update({f"openpnm_{k}": v for k, v in result.permeability.k_openpnm.items()})
    rows.append(row)
    print(f"  sample {sample_id}: {rock} {cube_size}³ saved -> {out_path.name}")

summary = pd.DataFrame(rows)
summary_path = OUT_DIR / f"network_{SPLIT}_summary.csv"
summary.to_csv(summary_path, index=False)
print(f"\nsummary saved: {summary_path}")
summary

---
## 4. Train GNN Permeability Model

Обучает PoreNetworkPermeabilityModel на извлечённых .pt сетях.
Цель — предсказать проницаемость k = (kx, ky, kz) на основе графовой структуры пор.
Сохраняет чекпоинт в `models/gnn_pnm_best.pth`.

In [ ]:
import torch.nn.functional as F
from tqdm import tqdm

from utils import PoreNetworkPermeabilityModel

In [ ]:
EPOCHS = 100
LR = 1e-3
HIDDEN = 64
LAYERS = 3
VAL_FRACTION = 0.2
PATIENCE = 15
SEED = 42
NETWORK_DIR = ROOT / "outputs" / "networks"
CHECKPOINT_PATH = ROOT / "models" / "gnn_pnm_best.pth"

In [ ]:
paths = sorted(NETWORK_DIR.glob(f"network_train_*.pt"))
if not paths:
    paths = sorted(NETWORK_DIR.glob("*.pt"))
if not paths:
    raise FileNotFoundError(f"No network .pt files found in {NETWORK_DIR}")
print("network files:", len(paths))

networks = [torch.load(path, map_location="cpu", weights_only=False) for path in paths]
first = networks[0]
node_in = first.node_attr.shape[1]
edge_in = first.edge_attr.shape[1]

generator = torch.Generator().manual_seed(SEED)
order = torch.randperm(len(networks), generator=generator).tolist()
val_count = max(1, int(round(len(networks) * VAL_FRACTION)))
val_count = min(val_count, len(networks) - 1) if len(networks) > 1 else 0

val_indices = set(order[:val_count])
train_networks = [n for idx, n in enumerate(networks) if idx not in val_indices]
val_networks = [n for idx, n in enumerate(networks) if idx in val_indices]

print("node_in:", node_in, "edge_in:", edge_in)
print("train networks:", len(train_networks), "val networks:", len(val_networks))

In [ ]:
model = PoreNetworkPermeabilityModel(node_in=node_in, edge_in=edge_in, hidden=HIDDEN, layers=LAYERS, mu=1.0e-3).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-5)
print("parameters:", sum(p.numel() for p in model.parameters()))

In [ ]:
def target_k_from_metadata(network):
    k = network.metadata.get("openpnm_k")
    if k is None:
        raise ValueError("No target k found. Add OpenPNM target to network.metadata['openpnm_k'].")
    return torch.tensor([k["kx"], k["ky"], k["kz"]], dtype=torch.float32)

def network_loss(network):
    network = network.to(device)
    target_k = target_k_from_metadata(network).to(device).clamp_min(1e-30)
    pred_k, log_g = model(
        network.node_attr,
        network.edge_index,
        network.edge_attr,
        network.coords,
        network.domain_size,
        log_g_hp=network.log_g_hp,
    )
    loss = F.smooth_l1_loss(torch.log(pred_k.clamp_min(1e-30)), torch.log(target_k))
    return loss, pred_k.detach(), target_k.detach(), log_g.detach()

In [ ]:
def evaluate_networks(split_networks):
    if not split_networks:
        return None, None, None
    model.eval()
    total_loss = 0.0
    last_pred, last_target = None, None
    with torch.no_grad():
        for network in split_networks:
            loss, pred_k, target_k, _ = network_loss(network)
            total_loss += float(loss.detach().cpu())
            last_pred = pred_k.detach().cpu()
            last_target = target_k.detach().cpu()
    return total_loss / len(split_networks), last_pred, last_target

best_loss = float("inf")
bad_epochs = 0
history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    last_train_pred = None
    last_train_target = None
    generator = torch.Generator().manual_seed(SEED + epoch)
    train_order = torch.randperm(len(train_networks), generator=generator).tolist()

    for idx in tqdm(train_order, desc=f"epoch {epoch}"):
        network = train_networks[idx]
        optimizer.zero_grad(set_to_none=True)
        loss, pred_k, target_k, _ = network_loss(network)
        loss.backward()
        optimizer.step()
        epoch_loss += float(loss.detach().cpu())
        last_train_pred = pred_k.detach().cpu()
        last_train_target = target_k.detach().cpu()
    train_loss = epoch_loss / max(len(train_networks), 1)
    val_loss, val_pred, val_target = evaluate_networks(val_networks)
    monitor_loss = val_loss if val_loss is not None else train_loss

    history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss})
    if val_loss is None:
        print(f"epoch={epoch} train_loss={train_loss:.6f}  last pred: {last_train_pred}, target: {last_train_target}")
    else:
        print(f"epoch={epoch} train_loss={train_loss:.6f} val_loss={val_loss:.6f}  val ex: pred={val_pred}, target={val_target}")

    if monitor_loss < best_loss:
        best_loss = monitor_loss
        bad_epochs = 0
        torch.save({
            "checkpoint_type": "gnn_pnm",
            "target_name": "openpnm_k",
            "model_state_dict": model.state_dict(),
            "node_in": node_in,
            "edge_in": edge_in,
            "hidden": HIDDEN,
            "layers": LAYERS,
            "epoch": epoch,
            "monitor_loss": monitor_loss,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "history": history,
            "train_files": [str(p.relative_to(ROOT)) for p in paths],
        }, CHECKPOINT_PATH)
        print("saved:", CHECKPOINT_PATH)
    else:
        bad_epochs += 1
        if bad_epochs >= PATIENCE:
            print(f"early stop at epoch={epoch}; best_loss={best_loss:.6f}")
            break

---
## 5. Run Full Pipeline

Загружает обученную сегментационную модель и GNN, прогоняет полный пайплайн
на одном кубе: сегментация → PoreSpy-экстракция → GNN-предсказание → проницаемость.

In [ ]:
import pandas as pd
from torch.utils.data import DataLoader

from utils import (
    DEFAULT_CUBE_SIZES,
    BereaPatchDataset,
    DigitalCorePipeline,
    TopologyAdaptiveRoutedUNet3D,
    PoreNetworkPermeabilityModel,
    TOPOLOGY_FEATURE_DIM,
)

In [ ]:
CUBE_SIZE = 128
VOXEL_SIZE_M = 2.25e-6
SEG_CKPT = ROOT / "models" / "segmentation_best.pth"
GNN_CKPT = ROOT / "models" / "gnn_pnm_best.pth"

In [ ]:
dataset = BereaPatchDataset(ROOT, split="val", cube_size=CUBE_SIZE, use_raw_gray=False, noise_types=["none"], balance=False, return_topology=True)
loader = DataLoader(dataset, batch_size=1, shuffle=False, num_workers=0)
batch = next(iter(loader))
cube = batch["x"][0].numpy()
rock = batch["rock"][0]
cube_size = int(batch["cube_size"][0])
DOMAIN_SIZE = (cube_size * VOXEL_SIZE_M, cube_size * VOXEL_SIZE_M, cube_size * VOXEL_SIZE_M)
print("rock:", rock, "cube:", cube.shape)

seg_checkpoint = torch.load(SEG_CKPT, map_location=device, weights_only=False)
ph_dim = int(seg_checkpoint.get("ph_dim", TOPOLOGY_FEATURE_DIM))
seg_model = TopologyAdaptiveRoutedUNet3D(
    base_channels=seg_checkpoint.get("base_channels", 16),
    ctx_dim=seg_checkpoint.get("ctx_dim", 64),
    ph_dim=ph_dim,
    topology_dim=ph_dim,
).to(device)
seg_model.load_state_dict(seg_checkpoint["model_state_dict"])

gnn_checkpoint = torch.load(GNN_CKPT, map_location=device, weights_only=False)
graph_model = PoreNetworkPermeabilityModel(
    node_in=gnn_checkpoint["node_in"],
    edge_in=gnn_checkpoint["edge_in"],
    hidden=gnn_checkpoint.get("hidden", 64),
    layers=gnn_checkpoint.get("layers", 3),
    mu=1.0e-3,
).to(device)
graph_model.load_state_dict(gnn_checkpoint["model_state_dict"])
print("models loaded")

In [ ]:
pipeline = DigitalCorePipeline(
    segmentation_model=seg_model,
    graph_model=graph_model,
    device=device,
    threshold=0.5,
    voxel_size=VOXEL_SIZE_M,
    mu=1.0e-3,
)

result = pipeline.run_cube(
    cube,
    input_is_pore_mask=False,
    domain_size=DOMAIN_SIZE,
    include_ph=True,
    compute_openpnm_baseline=True,
)

k_gnn = result.permeability.k_gnn_pnm.detach().cpu().numpy()
print("network:", result.network.metadata)
print("GNN+PNM k:", k_gnn)
print("OpenPNM k:", result.permeability.k_openpnm)

---
## 6. Validate Segmentation

Загружает обученный `TopologyAdaptiveRoutedUNet3D` и прогоняет на всей val-выборке.
Собирает Dice, BCE loss, voxel error rate с разбивкой по породе и размеру куба.

In [ ]:
import time
import warnings
from itertools import islice

import numpy as np
import pandas as pd

from torch.utils.data import DataLoader
from tqdm.notebook import tqdm

import matplotlib
matplotlib.use('inline')
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

In [ ]:
CHECKPOINT_PATH = ROOT / "models" / "segmentation_best.pth"
CUBE_SIZES = [64, 128]
BATCH_SIZE_BY_CUBE_SIZE = {64: 16, 128: 2, 192: 1}
VAL_SAMPLES_PER_GROUP = 16
NUM_WORKERS = 0
PIN_MEMORY = device == "cuda"
MAX_VAL_BATCHES = 8
USE_RAM_CACHE = True

In [ ]:
import utils.data as data_module
import utils.adaptive_routing as adaptive_module
import utils.losses as losses_module
import importlib

data_module = importlib.reload(data_module)
adaptive_module = importlib.reload(adaptive_module)
losses_module = importlib.reload(losses_module)

BereaPatchDataset = data_module.BereaPatchDataset
CubeSizeBatchSampler = data_module.CubeSizeBatchSampler
TopologyAdaptiveRoutedUNet3D = adaptive_module.TopologyAdaptiveRoutedUNet3D
dice_score_from_logits = losses_module.dice_score_from_logits

In [ ]:
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
base_channels = int(checkpoint.get("base_channels", 16))
ctx_dim = int(checkpoint.get("ctx_dim", 64))
ph_dim = int(checkpoint.get("ph_dim", 6))
print(f"Loaded: base_channels={base_channels}, ctx_dim={ctx_dim}, ph_dim={ph_dim}")
print(f"Best val_loss={checkpoint['val_loss']:.4f}, val_dice={checkpoint['val_dice']:.4f} at epoch={checkpoint['epoch']}")

model = TopologyAdaptiveRoutedUNet3D(
    in_channels=1, out_channels=1,
    base_channels=base_channels, ctx_dim=ctx_dim,
    ph_dim=ph_dim, topology_dim=ph_dim,
).to(device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
val_ds = BereaPatchDataset(
    ROOT, split="val", cube_size=CUBE_SIZES, use_raw_gray=False,
    noise_types=["none"], balance=False,
    samples_per_group=VAL_SAMPLES_PER_GROUP,
    return_aux_targets=False, return_topology=True,
)

class CachedPatchDataset:
    def __init__(self, dataset):
        import time
        self.samples = []
        start = time.perf_counter()
        for idx in tqdm(range(len(dataset)), desc="Caching val cubes", leave=False):
            sample = dataset[idx]
            self.samples.append({k: v.contiguous() if isinstance(v, torch.Tensor) else v for k, v in sample.items()})
        self.df = dataset.df.iloc[dataset.sample_index].reset_index(drop=True)
        self.sample_index = np.arange(len(self.samples))
        self.cube_sizes = dataset.cube_sizes
        elapsed = time.perf_counter() - start
        print(f"Cached {len(self.samples)} cubes in RAM in {elapsed:.1f}s")
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx): return self.samples[int(idx)]

if USE_RAM_CACHE:
    val_ds = CachedPatchDataset(val_ds)

val_sampler = CubeSizeBatchSampler(val_ds, BATCH_SIZE_BY_CUBE_SIZE, shuffle=False)
val_loader = DataLoader(val_ds, batch_sampler=val_sampler, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
print(f"Val dataset: {len(val_ds)} samples, {len(val_loader)} batches")
display(val_ds.df.groupby(["rock", "cube_size"]).size().rename("count").reset_index())

In [ ]:
results = []
bce_loss_fn = torch.nn.BCEWithLogitsLoss(reduction="none")

total_batches = len(val_loader) if MAX_VAL_BATCHES is None else min(len(val_loader), MAX_VAL_BATCHES)
val_iter = val_loader if MAX_VAL_BATCHES is None else islice(val_loader, MAX_VAL_BATCHES)
val_bar = tqdm(val_iter, total=total_batches, desc="Validation")

with torch.inference_mode():
    for batch in val_bar:
        x = batch["x"].to(device, non_blocking=PIN_MEMORY)
        y = batch["y"].to(device, non_blocking=PIN_MEMORY)
        ph_features = batch.get("ph_features")
        if ph_features is not None:
            ph_features = ph_features.to(device, non_blocking=PIN_MEMORY)

        out = model(x, ph_features=ph_features, return_dict=True)
        logits = out["logits"]
        prob = torch.sigmoid(logits)
        pred_mask = (prob >= 0.5).float()

        flat_pred = pred_mask.flatten(1)
        flat_y = y.flatten(1)
        intersection = (flat_pred * flat_y).sum(dim=1)
        denom = flat_pred.sum(dim=1) + flat_y.sum(dim=1)
        dice_values = ((2.0 * intersection + 1e-6) / (denom + 1e-6)).detach().cpu().tolist()
        bce_values = bce_loss_fn(logits, y).flatten(1).mean(dim=1).detach().cpu().tolist()
        error_values = (pred_mask != y).float().flatten(1).mean(dim=1).detach().cpu().tolist()

        rocks = list(batch["rock"]) if isinstance(batch["rock"], (list, tuple)) else [str(batch["rock"])] * x.size(0)
        cube_sizes = batch["cube_size"].detach().cpu().tolist()
        porosities = y.flatten(1).mean(dim=1).detach().cpu().tolist()

        for i in range(x.size(0)):
            results.append({
                "rock": rocks[i], "cube_size": int(cube_sizes[i]),
                "porosity": float(porosities[i]),
                "dice": float(dice_values[i]),
                "bce_loss": float(bce_values[i]),
                "error_rate": float(error_values[i]),
            })

df = pd.DataFrame(results)
print(f"\nProcessed {len(df)} cubes")
print(f"Mean Dice: {df['dice'].mean():.4f} ± {df['dice'].std():.4f}")
print(f"Mean error: {df['error_rate'].mean():.4f} ± {df['error_rate'].std():.4f}")

In [ ]:
print("DICE BY ROCK")
display(df.groupby("rock")["dice"].agg(["mean", "std", "min", "max", "count"]).round(4))
print("\nDICE BY CUBE SIZE")
display(df.groupby("cube_size")["dice"].agg(["mean", "std", "min", "max", "count"]).round(4))
print("\nDICE BY ROCK × CUBE SIZE")
display(df.groupby(["rock", "cube_size"])["dice"].agg(["mean", "std", "min", "max", "count"]).round(4))

In [ ]:
sns.set_style("whitegrid")
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

sns.histplot(df, x="dice", bins=30, ax=axes[0,0], kde=True)
axes[0,0].axvline(df["dice"].mean(), color="red", linestyle="--", label=f"mean={df['dice'].mean():.4f}")
axes[0,0].set_title("Dice distribution"); axes[0,0].legend()

sns.boxplot(data=df, x="rock", y="dice", ax=axes[0,1], palette="Set2")
axes[0,1].set_title("Dice by rock")

sns.boxplot(data=df, x="cube_size", y="dice", ax=axes[0,2], palette="Set3")
axes[0,2].set_title("Dice by cube size")

pivot = df.pivot_table(values="dice", index="rock", columns="cube_size", aggfunc="mean")
sns.heatmap(pivot, annot=True, fmt=".4f", cmap="RdYlGn", ax=axes[1,0], vmin=0.85, vmax=1.0)
axes[1,0].set_title("Mean Dice (rock × size)")

for rock_name in df["rock"].unique():
    subset = df[df["rock"] == rock_name]
    axes[1,1].scatter(subset["porosity"], subset["dice"], alpha=0.5, label=rock_name, s=30)
axes[1,1].set_xlabel("Porosity"); axes[1,1].set_ylabel("Dice"); axes[1,1].set_title("Dice vs Porosity"); axes[1,1].legend()

sns.histplot(df, x="error_rate", bins=30, ax=axes[1,2], kde=True, color="salmon")
axes[1,2].axvline(df["error_rate"].mean(), color="red", linestyle="--", label=f"mean={df['error_rate'].mean():.4f}")
axes[1,2].set_title("Error rate distribution"); axes[1,2].legend()

plt.tight_layout(); plt.show()

In [ ]:
TOP_K = 10
print("WORST CUBES")
display(df.nsmallest(TOP_K, "dice")[["rock", "cube_size", "dice", "error_rate", "porosity"]].reset_index(drop=True))
print("\nBEST CUBES")
display(df.nlargest(TOP_K, "dice")[["rock", "cube_size", "dice", "error_rate", "porosity"]].reset_index(drop=True))

In [ ]:
CSV_PATH = ROOT / "outputs" / "validation_results.csv"
df.to_csv(CSV_PATH, index=False)
print(f"Saved: {CSV_PATH}")
display(df.head())

---
## 7. (Optional) Compare Model Variants

Быстрое сравнение TopologyAdaptiveRoutedUNet3D vs AdaptiveRoutedUNet3D на 64³ кубах.
Полезно для выбора архитектуры перед полным обучением.

In [ ]:
from types import SimpleNamespace
from torch.utils.data import DataLoader
from tqdm import tqdm

from utils import (
    BCEDiceLoss,
    BereaPatchDataset,
    CubeSizeBatchSampler,
    TOPOLOGY_FEATURE_DIM,
    TopologyAdaptiveRoutedUNet3D,
    AdaptiveRoutedUNet3D,
    auxiliary_physics_loss,
    dice_score_from_logits,
    topology_prediction_loss,
)

In [ ]:
CUBE_SIZE = 64
EPOCHS = 100
SAMPLES_PER_GROUP = 4
MAX_TRAIN_BATCHES = 8
MAX_VAL_BATCHES = 4
BATCH_SIZE = 1
NUM_WORKERS = 0
BASE_CHANNELS = 8
CTX_DIM = 32
LR = 1.0e-4
AUX_WEIGHT = 0.05
TOPOLOGY_WEIGHT = 0.01
AMP = True
OUTPUT_CSV = ROOT / "outputs" / f"model_comparison_{CUBE_SIZE}.csv"
CHECKPOINT_DIR = ROOT / "models" / "notebook_runs"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
PIN_MEMORY = device == "cuda"

In [ ]:
def make_loader(dataset, *, shuffle: bool) -> DataLoader:
    sampler = CubeSizeBatchSampler(dataset, batch_size=BATCH_SIZE, shuffle=shuffle, seed=42 if shuffle else 43)
    kwargs = {"batch_sampler": sampler, "num_workers": NUM_WORKERS, "pin_memory": PIN_MEMORY}
    return DataLoader(dataset, **kwargs)

def router_entropy(alpha):
    probs = alpha.float().clamp_min(1.0e-8)
    return -(probs * probs.log()).sum(dim=-1).mean()

def run_epoch(model, loader, criterion, optimizer, scaler, *, train, max_batches):
    model.train(train)
    totals, counts = {}, {}
    desc = "train" if train else "val"
    iterator = tqdm(loader, desc=desc, leave=False)
    def update(name, value, n):
        totals[name] = totals.get(name, 0.0) + float(value) * n
        counts[name] = counts.get(name, 0) + n
    for batch_idx, batch in enumerate(iterator):
        if max_batches is not None and batch_idx >= max_batches:
            break
        x = batch["x"].to(device, non_blocking=True)
        y = batch["y"].to(device, non_blocking=True)
        porosity = batch["porosity"].to(device, non_blocking=True)
        percolates = batch["percolates"].to(device, non_blocking=True)
        ph_features = batch.get("ph_features"); topology_target = batch.get("topology_target")
        if ph_features is not None: ph_features = ph_features.to(device, non_blocking=True)
        if topology_target is not None: topology_target = topology_target.to(device, non_blocking=True)
        if train: optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(train), torch.amp.autocast(device_type=device.type, enabled=AMP and device.type == "cuda"):
            out = model(x, ph_features=ph_features, return_dict=True)
            logits = out["logits"]
            seg_loss, bce_loss, dice_loss = criterion(logits, y)
            aux_loss, _ = auxiliary_physics_loss(out, y, porosity_target=porosity, percolation_target=percolates, porosity_weight=AUX_WEIGHT, percolation_weight=AUX_WEIGHT)
            topo_loss, topo_parts = topology_prediction_loss(out, topology_target, topology_weight=TOPOLOGY_WEIGHT)
            loss = seg_loss + aux_loss + topo_loss
        if train:
            scaler.scale(loss).backward()
            scaler.step(optimizer); scaler.update()
        n = x.size(0)
        with torch.no_grad(): dice = dice_score_from_logits(logits, y)
        update("loss", loss.detach().cpu(), n); update("dice", dice.detach().cpu(), n)
        update("router_entropy", router_entropy(out["router_alpha"]).detach().cpu(), n)
        iterator.set_postfix({"loss": f"{totals['loss']/counts['loss']:.4f}", "dice": f"{totals['dice']/counts['dice']:.4f}"})
    return {name: totals[name]/counts[name] for name in totals}

In [ ]:
all_rows = []

for model_name, model_cls in [("adaptive", AdaptiveRoutedUNet3D), ("topology", TopologyAdaptiveRoutedUNet3D)]:
    print(f"\n=== {model_name} ===")
    common = dict(root_dir=ROOT, cube_size=[CUBE_SIZE], samples_per_group=SAMPLES_PER_GROUP, return_topology=(model_name == "topology"))
    train_ds = BereaPatchDataset(split="train", balance=True, **common)
    val_ds = BereaPatchDataset(split="val", balance=False, noise_types=["none"], **common)
    train_loader = make_loader(train_ds, shuffle=True)
    val_loader = make_loader(val_ds, shuffle=False)

    if model_name == "topology":
        model = model_cls(base_channels=BASE_CHANNELS, ctx_dim=CTX_DIM, ph_dim=TOPOLOGY_FEATURE_DIM, topology_dim=TOPOLOGY_FEATURE_DIM).to(device)
    else:
        model = model_cls(base_channels=BASE_CHANNELS, ctx_dim=CTX_DIM).to(device)

    criterion = BCEDiceLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1.0e-4)
    scaler = torch.amp.GradScaler(device.type, enabled=AMP and device.type == "cuda")
    best_val = float("inf")
    checkpoint_path = CHECKPOINT_DIR / f"{model_name}_quick_{CUBE_SIZE}.pth"

    for epoch in range(1, EPOCHS + 1):
        train_metrics = run_epoch(model, train_loader, criterion, optimizer, scaler, train=True, max_batches=MAX_TRAIN_BATCHES)
        val_metrics = run_epoch(model, val_loader, criterion, optimizer, scaler, train=False, max_batches=MAX_VAL_BATCHES)
        row = {"model": model_name, "epoch": epoch, "cube_size": CUBE_SIZE,
                **{f"train_{k}": v for k, v in train_metrics.items()},
                **{f"val_{k}": v for k, v in val_metrics.items()}}
        all_rows.append(row)
        if val_metrics["loss"] < best_val:
            best_val = val_metrics["loss"]
            torch.save({
                "model_state_dict": model.state_dict(), "model": model_name,
                "epoch": epoch, "val_loss": val_metrics["loss"], "val_dice": val_metrics["dice"],
                "base_channels": BASE_CHANNELS, "ctx_dim": CTX_DIM,
                "ph_dim": TOPOLOGY_FEATURE_DIM if model_name == "topology" else None,
                "topology_dim": TOPOLOGY_FEATURE_DIM if model_name == "topology" else None,
            }, checkpoint_path)
    if device == "cuda": torch.cuda.empty_cache()

summary = pd.DataFrame(all_rows)
summary.to_csv(OUTPUT_CSV, index=False)
print("\nComparison complete")
summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, metric, title in zip(axes, ["val_loss", "val_dice"], ["Validation Loss", "Validation Dice"]):
    for model_name in summary["model"].unique():
        df = summary[summary["model"] == model_name]
        ax.plot(df["epoch"], df[metric], label=model_name, linewidth=1.5)
    ax.set_xlabel("Epoch"); ax.set_title(title); ax.legend(); ax.grid(True, alpha=0.3)
fig.tight_layout(); plt.show()

best = summary.loc[summary.groupby("model")["val_dice"].idxmax()]
for _, row in best.iterrows():
    print(f"Best {row['model']}: epoch={row['epoch']}, val_loss={row['val_loss']:.4f}, val_dice={row['val_dice']:.4f}")